In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import trackio
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

keys = 'u', 'v', 'w', 'p_total'

def extract(real, keys):
    with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
        data = []
        for k in keys:
            data.append(np.array(f[real][k]))
        return np.stack(data, axis=1)

realizations = ['R13', 'R14', 'R15', 'R18', 'R19', 'R21', 'R22', 'R23', 'R26', 'R28', 'R3', 'R4', 'R5']
samplings = {real: np.s_[:-1:5000 // 300] if real in ["R3", "R4", "R5"] else np.s_[:] for real in realizations}

raw_d = {real: extract(real, keys)[samplings[real]] for real in realizations}
all_data = np.concatenate(list(raw_d.values()), axis=0)
T, C, X, Y, Z = all_data.shape
print("all_data", all_data.shape)

val_realizations = ["R18", "R5"]
test_realizations = ["R3", "R14"]  # ONLY USE FOR FINAL BENCHMARK, ANY DECISION TAKEN AFTER PERFORMANCE IS ESTABLISHED ON THIS DS IS LEAKAGE
train_realizations = [real for real in realizations if real not in val_realizations + test_realizations]
val_npds = np.concatenate([raw_d[r] for r in val_realizations])
train_npds = np.concatenate([raw_d[r] for r in train_realizations])

device = "cuda"

class DS(Dataset):
    def __init__(self, d):
        self.x = torch.from_numpy(d[:, 3:, :, 0, :]).float().to(device)
        self.y = torch.from_numpy(d[:, :3, :, :, :]).float().to(device)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

normalization_axes = (0, 2, 4,)
all_data_mean = all_data.mean(normalization_axes, keepdims=True)
all_data_std = all_data.std(normalization_axes, keepdims=True) + 1e-10

all_data_mean_gpu = torch.from_numpy(all_data_mean).to(device)
all_data_std_gpu = torch.from_numpy(all_data_std).to(device)

def normalize(data):
    return (data - all_data_mean) / all_data_std

def unnormalize_velocity_gpu(data):
    return (data * all_data_std_gpu[:, :3]) + all_data_mean_gpu[:, :3]

val_ds = DS(normalize(val_npds))
train_ds = DS(normalize(train_npds))

def predict(model, flat_x_ds):
    model.eval()
    with torch.no_grad():
        return model(flat_x_ds)

batch_size = 4
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)
print("train", len(train_dl), "val", len(val_dl))


all_data (3900, 4, 24, 33, 24)
train 675 val 150


In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding="same"),
            nn.SiLU(),
            nn.Conv3d(out_ch, out_ch, 3, padding="same"),
            nn.SiLU(),
        )
    def forward(self, x):
        return self.conv(x)


class M4(nn.Module):
    """Pressure -> velocity network with a WIDTH-PARAMETERIZED lift.

    lift_width : channels produced by the lift conv (was hardcoded 32*D).
                 This is the "width of the lift" we want to sweep.
    c3d        : channels of the 3D conv blocks that refine the lifted field.
    n_blocks   : number of 3D ConvBlocks.
    widen      : if >0, insert a Conv2d(widthen)->widthen pre-lift block so the
                 lift reads from a wider 2D representation (tests whether the
                 bottleneck is the 2D->3D jump vs the lift conv itself).
    """
    def __init__(self, lift_width=32, c3d=32, n_blocks=5, widen=0):
        super().__init__()
        self.D = 33
        self.c3d = c3d
        self.xd, self.zd = 24, 24

        self.pressure_encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding="same"),
            nn.SiLU(),
            nn.Conv2d(8, 32, 3, padding="same"),
            nn.SiLU(),
        )

        # optional pre-lift widening of the 2D representation
        in_lift = 32
        if widen > 0:
            self.pre_widen = nn.Sequential(
                nn.Conv2d(32, widen, 3, padding="same"),
                nn.SiLU(),
            )
            in_lift = widen

        # THE LIFT: 2D pressure features -> 3D velocity field (D levels)
        self.lift = nn.Conv2d(in_lift, self.D * lift_width, 3, padding="same")

        # project lift_width -> c3d if they differ
        self.lift_proj = (nn.Conv3d(lift_width, c3d, 1)
                          if lift_width != c3d else nn.Identity())

        self.conv_blocks = nn.Sequential(*[ConvBlock(c3d, c3d) for _ in range(n_blocks)])
        self.output = nn.Conv3d(c3d, 3, 3, padding="same")

    def forward(self, pressure):
        x = self.pressure_encoder(pressure)
        if hasattr(self, "pre_widen"):
            x = self.pre_widen(x)
        x = self.lift(x)
        # lift.out_channels == D * lift_width  ->  [B, D*lift_width, xd, zd]
        # reshape to [B, lift_width, xd, D, zd]
        x = x.reshape(-1, self.lift.out_channels // self.D, self.xd, self.D, self.zd)
        x = self.lift_proj(x)
        x = self.conv_blocks(x)
        x = self.output(x)
        return x


def make_model(lift_width=32, c3d=32, n_blocks=5, widen=0):
    m = M4(lift_width=lift_width, c3d=c3d, n_blocks=n_blocks, widen=widen).to(device)
    n = sum(p.numel() for p in m.parameters())
    print(f"lift_width={lift_width} c3d={c3d} n_blocks={n_blocks} widen={widen} -> {n/1e6:.2f}M params")
    return m


In [3]:
# ---------------------------------------------------------------------------
# Weighted spectral loss  (L = MSE + lambda * L_spectral)
#   L_spectral = sum_k W[k]|U_k - V_k|^2 / sum_k W[k]
#   W[k] : fixed equal-per-log-band weight w_b = 1/(N_b*E_b), E_b from TARGET
# ---------------------------------------------------------------------------

class SpectralLoss(nn.Module):
    def __init__(self, target, dim=(-2, -1), n_bands=6, kmin=1.0, lambda_weight=1.0):
        super().__init__()
        assert lambda_weight >= 0
        self.dim = dim
        self.lambda_weight = lambda_weight
        self.W = self._build_weights(target, dim, n_bands, kmin)   # [1,1,X,1,Z]

    def _build_weights(self, target, dim, n_bands, kmin):
        with torch.no_grad():
            target = target.detach()
            spec = torch.fft.fft2(target, dim=dim, norm="ortho")
            E = (spec.abs() ** 2).mean(dim=(0, 1, 3))             # [X, Z]
            Xn, Zn = E.shape
            kx = torch.fft.fftfreq(Xn, device=E.device) * (2 * np.pi)
            kz = torch.fft.fftfreq(Zn, device=E.device) * (2 * np.pi)
            KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
            k_mag = torch.hypot(KX, KZ)
            k_max = k_mag.max()
            edges = torch.logspace(torch.log10(torch.tensor(kmin)),
                                   torch.log10(k_max), n_bands + 1, device=E.device)
            W = torch.zeros_like(k_mag)
            band_E, band_N = [], []
            for b in range(n_bands):
                lo, hi = float(edges[b]), float(edges[b + 1])
                mask = (k_mag >= lo) & (k_mag < hi)
                N_b = max(int(mask.sum()), 1)
                E_b = E[mask].mean().clamp(min=1e-12)
                W[mask] = 1.0 / (N_b * E_b)
                band_E.append(float(E_b)); band_N.append(N_b)
            W[k_mag == 0] = 0.0
            self.band_info = list(zip([f"{lo:.2f}-{hi:.2f}" for lo, hi in zip(edges[:-1], edges[1:])], band_N, band_E))
            return W[None, None, :, None, :]

    def forward(self, pred, target):
        spec_pred = torch.fft.fft2(pred, dim=self.dim, norm="ortho")
        spec_tgt = torch.fft.fft2(target, dim=self.dim, norm="ortho")
        diff2 = (spec_pred - spec_tgt).abs() ** 2
        weighted = (diff2 * self.W).sum() / self.W.sum()
        mse = torch.nn.functional.mse_loss(pred, target)
        return mse + self.lambda_weight * weighted, mse, weighted


xb, yb = next(iter(val_dl))
spectral = SpectralLoss(yb, dim=(-2, -1), n_bands=6, kmin=1.0, lambda_weight=1.0).to(device)
print("W shape:", tuple(spectral.W.shape))
print(f"{'band |k|':>14}  {'#modes':>7}  {'mean target E':>14}  {'w_b':>10}")
for name, N_b, E_b in spectral.band_info:
    print(f"{name:>14}  {N_b:>7d}  {E_b:>14.3e}  {1.0/(N_b*E_b):>10.3e}")


W shape: (1, 1, 24, 1, 24)
      band |k|   #modes   mean target E         w_b
     1.00-1.28       24       2.238e+00   1.861e-02
     1.28-1.64       52       1.702e+00   1.130e-02
     1.64-2.11       76       1.039e+00   1.267e-02
     2.11-2.70      144       1.010e+00   6.872e-03
     2.70-3.47      170       5.961e-01   9.868e-03
     3.47-4.44       64       1.445e-03   1.081e+01


In [8]:
# ---------------------------------------------------------------------------
# Config  (the sweepable knobs are lift_width / c3d / n_blocks / widen / spectral_lambda)
# ---------------------------------------------------------------------------
epochs = 20
lr = 5e-4
optim = torch.optim.Adam
batch_size = 4
project_name = "width-study"   # constant across the sweep; each run is distinguished by its config

# --- architecture knobs (sweep these) ---
lift_width = 32      # width of the lift conv  (the main thing we want to analyze)
c3d = 32             # 3D conv-block channels
n_blocks = 5
widen = 0            # >0 : pre-lift 2D widening

# --- spectral loss ---
spectral_lambda = 0.2

model = make_model(lift_width=lift_width, c3d=c3d, n_blocks=n_blocks, widen=widen)
train_loss = SpectralLoss(yb, dim=(-2, -1), n_bands=6, kmin=1.0,
                          lambda_weight=spectral_lambda).to(device)

# fixed band edges for diagnostics
XN, ZN = 24, 24
_kx = torch.fft.fftfreq(XN) * (2 * np.pi)
_kz = torch.fft.fftfreq(ZN) * (2 * np.pi)
_KX, _KZ = torch.meshgrid(_kx, _kz, indexing="ij")
_KMAG = torch.hypot(_KX, _KZ)
BAND_EDGES = torch.logspace(torch.log10(torch.tensor(1.0)),
                            torch.log10(_KMAG.max()), 7).tolist()
print("bands:", [f"{BAND_EDGES[i]:.2f}-{BAND_EDGES[i+1]:.2f}" for i in range(6)])
print(f"train loss = MSE + {spectral_lambda} * spectral")


lift_width=32 c3d=32 n_blocks=5 widen=0 -> 0.59M params
bands: ['1.00-1.28', '1.28-1.64', '1.64-2.11', '2.11-2.70', '2.70-3.47', '3.47-4.44']
train loss = MSE + 0.2 * spectral


Task was destroyed but it is pending!
task: <Task pending name='Task-49' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /data3/chancrin/code/flow-diffuse/.venv/lib/python3.10/site-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-50' coro=<_async_in_context.<locals>.preserve_context() running at /data3/chancrin/code/flow-diffuse/.venv/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /data3/chancrin/code/flow-diffuse/.venv/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
/data3/chancrin/code/flow-diffuse/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py:513: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  super().__setattr__("_load_state_dict_pre_hooks", OrderedDict())
Task was destroyed but it is pending!
task: <Task pending name='Task-50' coro=<_async_in_context.<locals>.preserve_context() runni

In [9]:
# ---------------------------------------------------------------------------
# Train / eval / full-val spectral + per-band-skill diagnostics
# ---------------------------------------------------------------------------

def train(model, opti, loss_fn, ds):
    model.train()
    tot = mse_tot = spec_tot = 0.0
    for x, y in ds:
        opti.zero_grad()
        total, mse, spec = loss_fn(model(x), y)
        total.backward()
        opti.step()
        tot += total.item(); mse_tot += mse.item(); spec_tot += spec.item()
    n = len(ds)
    return tot / n, mse_tot / n, spec_tot / n


@torch.no_grad()
def evaluate(model, ds):
    model.eval()
    tot = 0.0
    for x, y in ds:
        tot += torch.nn.functional.mse_loss(model(x), y).item()
    return tot / len(ds)


@torch.no_grad()
def val_spectral_bands(model, ds, edges=BAND_EDGES, dim=(-2, -1)):
    """Per-band SHARE of total val error energy (where the error lives)."""
    model.eval()
    Xn, Zn = ds.dataset.y.shape[2], ds.dataset.y.shape[4]
    kx = torch.fft.fftfreq(Xn, device=device) * (2 * np.pi)
    kz = torch.fft.fftfreq(Zn, device=device) * (2 * np.pi)
    KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
    k_mag = torch.hypot(KX, KZ).to(device)
    band_err = [0.0] * (len(edges) - 1)
    for x, y in ds:
        E = (torch.fft.fft2(model(x) - y, dim=dim, norm="ortho").abs() ** 2).mean(dim=(0, 1, 3))
        for b in range(len(edges) - 1):
            mask = (k_mag >= edges[b]) & (k_mag < edges[b + 1])
            band_err[b] += float(E[mask].sum())
    tot = sum(band_err)
    return {f"{edges[b]:.2f}-{edges[b+1]:.2f}": 100.0 * band_err[b] / tot
            for b in range(len(edges) - 1)}


@torch.no_grad()
def val_band_skill(model, ds, edges=BAND_EDGES, dim=(-2, -1)):
    """Per-band skill = 1 - E_err(b)/E_target(b)  (== EV restricted to band b).
    skill ~ 1 : good OR trivial-zero (check E_pred)   |   skill < 0 : worse than mean
    """
    model.eval()
    Xn, Zn = ds.dataset.y.shape[2], ds.dataset.y.shape[4]
    kx = torch.fft.fftfreq(Xn, device=device) * (2 * np.pi)
    kz = torch.fft.fftfreq(Zn, device=device) * (2 * np.pi)
    KX, KZ = torch.meshgrid(kx, kz, indexing="ij")
    k_mag = torch.hypot(KX, KZ).to(device)
    band_err = [0.0] * (len(edges) - 1)
    band_tgt = [0.0] * (len(edges) - 1)
    for x, y in ds:
        pred = model(x)
        E_err = (torch.fft.fft2(pred - y, dim=dim, norm="ortho").abs() ** 2).mean(dim=(0, 1, 3))
        E_tgt = (torch.fft.fft2(y, dim=dim, norm="ortho").abs() ** 2).mean(dim=(0, 1, 3))
        for b in range(len(edges) - 1):
            mask = (k_mag >= edges[b]) & (k_mag < edges[b + 1])
            band_err[b] += float(E_err[mask].sum())
            band_tgt[b] += float(E_tgt[mask].sum())
    return {f"{edges[b]:.2f}-{edges[b+1]:.2f}": 1.0 - band_err[b] / band_tgt[b]
            for b in range(len(edges) - 1)}


def train_loop(model, loss_fn, ds_train, ds_val, epochs, optim, lr, project_name, run_config=None):
    """project_name : the STUDY name (constant across the sweep, e.g. 'width-study').
    run_config     : per-run params (lift_width, c3d, ...) -> stored as run config so
                     each sweep point is a distinct run under the same project.
                     trackio auto-names the run if none is given."""
    opti = optim(model.parameters(), lr=lr)
    cfg = {"epochs": epochs, "learning_rate": lr, "spectral_lambda": loss_fn.lambda_weight,
           "lift_width": model.lift.out_channels // model.D, "c3d": model.c3d,
           "n_blocks": len(model.conv_blocks), "widen": getattr(model, "pre_widen", None) is not None}
    if run_config:
        cfg.update(run_config)
    trackio.init(project=project_name, config=cfg)
    with torch.no_grad():
        base_bands = val_spectral_bands(model, ds_val)
        base_skill = val_band_skill(model, ds_val)
    trackio.log({"epoch": -1,
                 **{f"val/err_share_%/{n}": v for n, v in base_bands.items()},
                 **{f"val/skill/{n}": v for n, v in base_skill.items()}})

    for epoch in range(epochs):
        train_err, train_mse, train_spec = train(model, opti, loss_fn, ds_train)
        val_mse = evaluate(model, ds_val)
        val_ev = 1.0 - val_mse
        bands = val_spectral_bands(model, ds_val)
        skill = val_band_skill(model, ds_val)
        trackio.log({
            "epoch": epoch,
            "train/loss": train_err, "train/mse": train_mse, "train/spectral": train_spec,
            "val/mse": val_mse, "val/EV": val_ev,
            **{f"val/err_share_%/{name}": v for name, v in bands.items()},
            **{f"val/skill/{name}": v for name, v in skill.items()},
        })
    trackio.finish()


In [ ]:
# ---------------------------------------------------------------------------
# Run
#   project_name = the STUDY (constant across the sweep)
#   the architecture (lift_width, ...) is stored as the RUN config, so each
#   sweep point is a distinct, comparable run under the same project.
# ---------------------------------------------------------------------------
n_epochs = 20   # drop to 2 for a smoke test first

train_loop(
    model=model,
    loss_fn=train_loss,
    ds_train=train_dl,
    ds_val=val_dl,
    epochs=n_epochs,
    optim=optim,
    lr=lr,
    project_name=project_name,          # e.g. "width-study"
    run_config={                        # what distinguishes THIS run
        "lift_width": lift_width,
        "c3d": c3d,
        "n_blocks": n_blocks,
        "widen": widen,
        "spectral_lambda": spectral_lambda,
    },
)
